
# Melt-Pool Shape Classification Using Deep Learning

This notebook builds a multiclass deep-learning classifier for the `meltpool shape` target in the supplied dataset.

**Model:** PyTorch multilayer perceptron (MLP)  
**Task:** Five-class classification — LOF, balling, desirable, keyhole, and spatter formation.

## Important dataset considerations

- The dataset contains 1,249 labelled observations.
- `spatter formation` has only 8 observations. Class-weighted training helps, but no model can learn this class reliably from so few examples. Its test metrics will be highly uncertain.
- Only raw process settings and material properties are used as predictors.
- Publication identifiers, comments, unnamed columns, porosity/relative-density outcomes, and derived process ratios are excluded to reduce target leakage.
- `Hatch spacing` is excluded automatically if more than 80% of its values are missing.
- The split is stratified. Because the rare class is very small, use the per-class support counts when interpreting the results.

Run the notebook from top to bottom. In Google Colab, upload the CSV when prompted; locally, place the CSV beside the notebook or enter its path.


In [ ]:

# If needed, install dependencies in a fresh environment:
# %pip install pandas numpy matplotlib scikit-learn torch joblib

import copy
import json
import random
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)


In [ ]:

# Load the dataset.
# Option 1: place the CSV beside this notebook.
# Option 2: upload it when using Google Colab.
csv_path = Path("meltpoolnet_classification(1).csv")

if not csv_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file was uploaded.")
        csv_path = Path(next(iter(uploaded.keys())))
    except ImportError:
        entered_path = input("Enter the full path to the melt-pool CSV file: ").strip()
        csv_path = Path(entered_path)

if not csv_path.exists():
    raise FileNotFoundError(f"Could not find the dataset at: {csv_path}")

df = pd.read_csv(csv_path)
print("Loaded:", csv_path)
print("Dataset shape:", df.shape)
display(df.head())


## 1. Initial dataset inspection

In [ ]:

TARGET = "meltpool shape"

if TARGET not in df.columns:
    raise KeyError(f"Target column '{TARGET}' was not found. Available columns: {df.columns.tolist()}")

df[TARGET] = df[TARGET].astype("string").str.strip()
df = df.dropna(subset=[TARGET]).copy()

print("Rows and columns:", df.shape)
print("Duplicate full rows:", df.duplicated().sum())
print("\nTarget class counts:")
display(df[TARGET].value_counts().rename_axis("Class").to_frame("Observations"))

ax = df[TARGET].value_counts().plot(kind="bar", figsize=(8, 4))
ax.set_title("Melt-pool class distribution")
ax.set_xlabel("Melt-pool class")
ax.set_ylabel("Number of observations")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

missing_table = (
    df.isna().mean().mul(100).sort_values(ascending=False)
      .rename("Missing (%)").to_frame()
)
display(missing_table.head(20).round(1))



## 2. Feature selection and leakage control

The predictors below are process parameters and material properties that could reasonably be available before or during a build.

**Excluded intentionally:**
- `meltpool shape` is the target.
- `comment` and `Unnamed` columns may contain class descriptions or poorly documented fields.
- `paper`, `paper ID`, and source URLs are publication metadata, not process predictors.
- `porosity` and `relative density` are quality outcomes that may be measured after processing.
- `p/lb`, `p/l`, `p/b2`, `p/b`, `vb`, and `vl` are derived ratios/indicators. They are excluded from this first model to reduce the risk of learning rule-based proxies for the target rather than learning from raw inputs.
- `Process` is constant in this dataset and therefore carries no predictive information.

Numeric columns are parsed with `errors="coerce"` so values such as `?`, `#DIV/0!`, and `#VALUE!` become missing values. Missing numeric values are imputed using training-set medians, with missingness indicators added by the imputer.


In [ ]:

categorical_candidates = ["Material", "Sub-process"]

numeric_candidates = [
    "Power",
    "Velocity",
    "Hatch spacing",
    "layer thickness",
    "beam D",
    "density",
    "Cp",
    "k",
    "melting T",
    "absorption coefficient",
    "minimal absorptivity",
    "Zn (wt%)",
    "Mg (wt%)",
    "Si (wt%)",
    "Al (wt%)",
    "Sn (wt%)",
    "Zr (wt%)",
    "W (wt%)",
    "Ti (wt%)",
    "V (wt%)",
    "Co (wt%)",
    "Cu (wt%)",
    "Ta (wt%)",
    "Nb (wt%)",
    "Ni (wt.%)",
    "Cr (wt.%)",
    "Fe (wt.%)",
    "Mn (wt%)",
    "Mo (wt.%)",
]

categorical_features = [c for c in categorical_candidates if c in df.columns]
numeric_features = [c for c in numeric_candidates if c in df.columns]

if not categorical_features:
    raise ValueError("No expected categorical predictors were found.")
if not numeric_features:
    raise ValueError("No expected numeric predictors were found.")

# Convert candidate numeric predictors safely.
for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove numeric predictors with more than 80% missingness or no observed values.
missing_rates = df[numeric_features].isna().mean()
dropped_high_missing = [
    col for col in numeric_features
    if missing_rates[col] > 0.80 or df[col].notna().sum() == 0
]
numeric_features = [c for c in numeric_features if c not in dropped_high_missing]

print("Categorical features:", categorical_features)
print("Numeric features retained:", numeric_features)
print("Numeric features dropped for >80% missingness / all-missing:", dropped_high_missing)
print("Total raw predictors:", len(categorical_features) + len(numeric_features))

X = df[categorical_features + numeric_features].copy()
y_text = df[TARGET].astype(str).str.strip()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
class_names = label_encoder.classes_.tolist()

print("\nEncoded classes:")
for idx, name in enumerate(class_names):
    print(f"{idx}: {name}")

class_counts = y_text.value_counts()
if class_counts.min() < 5:
    print("\nWARNING: At least one class has fewer than 5 observations. "
          "Stratified holdout metrics and model learning for that class will be unstable.")



## 3. Stratified train/validation/test split

The split uses approximately 66% training, 12% validation, and 22% test data. The slightly larger test portion is intentional so the very small spatter class is represented by more than one test observation in this dataset.

The test set is held back until final evaluation. Preprocessing is fitted on the training data only.


In [ ]:

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.22, random_state=SEED, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15,
    random_state=SEED, stratify=y_train_val
)

def show_split_counts(name, y_part):
    counts = pd.Series(label_encoder.inverse_transform(y_part)).value_counts()
    return counts.rename(name)

split_counts = pd.concat(
    [
        show_split_counts("Train", y_train),
        show_split_counts("Validation", y_val),
        show_split_counts("Test", y_test),
    ],
    axis=1
).fillna(0).astype(int)

display(split_counts)
print(f"Train: {len(y_train)} | Validation: {len(y_val)} | Test: {len(y_test)}")



## 4. Preprocessing

- Numeric features: median imputation, missingness indicators, and standardisation.
- Categorical features: most-frequent imputation and one-hot encoding.
- The preprocessing pipeline is fitted only on the training split to prevent information leakage.


In [ ]:

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

def to_dense_float32(matrix):
    if hasattr(matrix, "toarray"):
        matrix = matrix.toarray()
    return np.asarray(matrix, dtype=np.float32)

X_train_np = to_dense_float32(preprocessor.fit_transform(X_train))
X_val_np = to_dense_float32(preprocessor.transform(X_val))
X_test_np = to_dense_float32(preprocessor.transform(X_test))

print("Processed train shape:", X_train_np.shape)
print("Processed validation shape:", X_val_np.shape)
print("Processed test shape:", X_test_np.shape)



## 5. Define the deep learning model

A compact multilayer perceptron is used for this tabular dataset. Dropout and weight decay help reduce overfitting. The output layer has one neuron per melt-pool class.

Class weights are calculated using the inverse square root of the training class counts. This gives additional importance to minority classes without applying the very large weights that a direct inverse-frequency method would produce for the 8-row spatter class.


In [ ]:

class MeltPoolMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.network(x)


num_classes = len(class_names)
model = MeltPoolMLP(
    input_dim=X_train_np.shape[1],
    num_classes=num_classes
).to(DEVICE)

train_class_counts = np.bincount(y_train, minlength=num_classes)
class_weights_np = 1.0 / np.sqrt(train_class_counts.astype(float))
class_weights_np = class_weights_np / class_weights_np.mean()
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=DEVICE)

print("Training class counts:", dict(zip(class_names, train_class_counts)))
print("Class weights:", dict(zip(class_names, class_weights_np.round(3))))

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

train_dataset = TensorDataset(
    torch.tensor(X_train_np, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long),
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)

X_val_tensor = torch.tensor(X_val_np, dtype=torch.float32, device=DEVICE)
y_val_tensor = torch.tensor(y_val, dtype=torch.long, device=DEVICE)


## 6. Train with early stopping

In [ ]:

MAX_EPOCHS = 250
PATIENCE = 30
best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
best_state = None

history = {
    "train_weighted_loss": [],
    "validation_loss": [],
    "train_accuracy": [],
    "validation_accuracy": [],
}

start_time = time.time()

for epoch in range(MAX_EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * len(batch_y)
        correct += (logits.argmax(dim=1) == batch_y).sum().item()
        total += len(batch_y)

    train_loss = running_loss / total
    train_accuracy = correct / total

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_tensor)
        # Unweighted validation cross-entropy is used for early stopping.
        val_loss = nn.functional.cross_entropy(val_logits, y_val_tensor).item()
        val_accuracy = (
            (val_logits.argmax(dim=1) == y_val_tensor).float().mean().item()
        )

    history["train_weighted_loss"].append(train_loss)
    history["validation_loss"].append(val_loss)
    history["train_accuracy"].append(train_accuracy)
    history["validation_accuracy"].append(val_accuracy)

    if val_loss < best_val_loss - 1e-5:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch+1:03d} | "
            f"train weighted loss={train_loss:.4f} | "
            f"validation loss={val_loss:.4f} | "
            f"train acc={train_accuracy:.3f} | "
            f"validation acc={val_accuracy:.3f}"
        )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}.")
        break

if best_state is None:
    raise RuntimeError("Training did not produce a valid model checkpoint.")

model.load_state_dict(best_state)
training_seconds = time.time() - start_time

print(f"\nTraining completed in {training_seconds:.1f} seconds.")
print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.4f}")



## 7. Training curves

The training objective is class-weighted, while the validation loss is unweighted. They are therefore useful for monitoring their own trends, but their absolute values should not be compared directly.


In [ ]:

history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df["train_weighted_loss"], label="Training weighted loss")
axes[0].plot(history_df["validation_loss"], label="Validation loss")
axes[0].set_title("Loss curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history_df["train_accuracy"], label="Training accuracy")
axes[1].plot(history_df["validation_accuracy"], label="Validation accuracy")
axes[1].set_title("Accuracy curves")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Final evaluation on the untouched test set

In [ ]:

model.eval()
X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32, device=DEVICE)

with torch.no_grad():
    test_logits = model(X_test_tensor)
    test_probabilities = torch.softmax(test_logits, dim=1).cpu().numpy()

y_pred = test_probabilities.argmax(axis=1)

test_accuracy = accuracy_score(y_test, y_pred)
test_balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
test_macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
test_weighted_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Test accuracy:          {test_accuracy:.4f}")
print(f"Balanced accuracy:      {test_balanced_accuracy:.4f}")
print(f"Macro F1-score:          {test_macro_f1:.4f}")
print(f"Weighted F1-score:      {test_weighted_f1:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=np.arange(num_classes),
        target_names=class_names,
        zero_division=0,
        digits=3,
    )
)

cm = confusion_matrix(y_test, y_pred, labels=np.arange(num_classes))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
).plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
ax.set_title("Test-set confusion matrix")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

print("Test-set class support:")
display(
    pd.Series(label_encoder.inverse_transform(y_test))
      .value_counts()
      .reindex(class_names, fill_value=0)
      .rename("Test observations")
      .to_frame()
)



## 9. Compare with a simple majority-class baseline

A deep learning model should be compared with a simple reference. The majority-class baseline always predicts the most common training class.


In [ ]:

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

baseline_comparison = pd.DataFrame(
    {
        "Model": ["Majority-class baseline", "PyTorch MLP"],
        "Accuracy": [
            accuracy_score(y_test, dummy_pred),
            test_accuracy,
        ],
        "Balanced accuracy": [
            balanced_accuracy_score(y_test, dummy_pred),
            test_balanced_accuracy,
        ],
        "Macro F1": [
            f1_score(y_test, dummy_pred, average="macro", zero_division=0),
            test_macro_f1,
        ],
        "Weighted F1": [
            f1_score(y_test, dummy_pred, average="weighted", zero_division=0),
            test_weighted_f1,
        ],
    }
)

display(baseline_comparison.round(4))



## 10. Predict the melt-pool class for new process parameters

Supply a pandas DataFrame with the same raw predictor columns used above. If an optional field is absent, it is filled with a missing value and handled by the fitted preprocessing pipeline. `Material` and `Sub-process` are particularly useful categorical inputs.


In [ ]:

def predict_meltpool(new_data: pd.DataFrame) -> pd.DataFrame:
    new_data = new_data.copy()

    required_features = categorical_features + numeric_features
    for col in required_features:
        if col not in new_data.columns:
            new_data[col] = np.nan

    for col in numeric_features:
        new_data[col] = pd.to_numeric(new_data[col], errors="coerce")

    transformed = to_dense_float32(
        preprocessor.transform(new_data[required_features])
    )

    model.eval()
    with torch.no_grad():
        logits = model(
            torch.tensor(transformed, dtype=torch.float32, device=DEVICE)
        )
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()

    predicted_indices = probabilities.argmax(axis=1)
    result = pd.DataFrame(
        {
            "Predicted melt-pool shape": [
                class_names[i] for i in predicted_indices
            ],
            "Prediction confidence": probabilities.max(axis=1),
        }
    )

    for i, class_name in enumerate(class_names):
        result[f"Probability: {class_name}"] = probabilities[:, i]

    return result


# Demonstration using one held-out test observation.
# Replace this with a DataFrame containing new experimental process parameters.
example_new_input = X_test.iloc[[0]].copy()
display(example_new_input)
display(predict_meltpool(example_new_input).round(4))


## 11. Save the trained model and preprocessing objects

In [ ]:

model_path = Path("meltpool_mlp_model.pth")
preprocessor_path = Path("meltpool_preprocessor.joblib")
label_encoder_path = Path("meltpool_label_encoder.joblib")

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "input_dim": int(X_train_np.shape[1]),
        "num_classes": int(num_classes),
        "class_names": class_names,
        "categorical_features": categorical_features,
        "numeric_features": numeric_features,
        "seed": SEED,
    },
    model_path,
)

joblib.dump(preprocessor, preprocessor_path)
joblib.dump(label_encoder, label_encoder_path)

print("Saved files:")
print(model_path.resolve())
print(preprocessor_path.resolve())
print(label_encoder_path.resolve())



## 12. Interpretation and limitations

1. **Rare class:** only 8 observations are labelled `spatter formation`. Even if the test-set score is perfect, it is based on only a few examples and is not strong evidence of reliable spatter detection.
2. **Literature-derived dataset:** a random row-wise split may place observations from the same paper or experimental campaign in both training and test sets. This can make performance look better than performance on a genuinely unseen publication or machine.
3. **Recommended validation for a publication:** create a reliable source-paper/group identifier and use a group-aware split (for example, `GroupKFold` or `GroupShuffleSplit`) so that all observations from a paper are held in one split. The source metadata in this CSV is incomplete/irregular, so it should be cleaned before group-based validation.
4. **External validation:** test the final model on new melt-pool observations from an independent experiment or publication.
5. **Model development:** compare this MLP with tree-based models (Random Forest, XGBoost/CatBoost) and report repeated stratified or group-aware cross-validation. Do not select a model using the final test set.
6. **Scientific use:** the output is a data-driven prediction, not a physical guarantee. Use it alongside process knowledge and experimental validation.
